## 0. Installation and imports

In [1]:
import subprocess, sys
try:
    import timm
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'timm>=0.9.12'])

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f'Ambiente: {"Google Colab" if IN_COLAB else "Local"}')

In [3]:
import json
import random
import time
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torchvision.transforms as T
import torchvision.models as tvm
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import timm

from PIL import Image
from sklearn.metrics import f1_score, precision_score, recall_score, average_precision_score, roc_auc_score

print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'timm    : {timm.__version__}')
from tqdm.auto import tqdm
import datetime
import sys

## 1. Path and hyperparameters configuration

In [4]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive/Trilha3-V2')
    IMGS_DIR = Path('/content/Imgs')
else:
    BASE_DIR = Path('e:/Doutorado-V3/Trilha3-V2')
    IMGS_DIR = Path('e:/Doutorado-V3/Data/Imgs')

SPLITS_DIR  = BASE_DIR / 'splits'
RESULTS_DIR = BASE_DIR / 'results' / 'baseline'
CKPT_DIR    = BASE_DIR / 'checkpoints'
FIGS_DIR    = BASE_DIR / 'figs' / 'training'

for d in [RESULTS_DIR, CKPT_DIR, FIGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CORE_LABELS = ['ENANTEMA', 'PÓLIPO', 'ÚLCERA', 'EROSÃO', 'MICRONODULARIDADE']
N_FOLDS     = 5
N_CLASSES   = len(CORE_LABELS)

IMG_SIZE    = 224
BATCH_SIZE  = 128
MAX_EPOCHS  = 50
LR_BACKBONE = 1e-4
LR_HEAD     = 1e-3
WEIGHT_DECAY= 1e-4
ES_PATIENCE = 10
LR_PATIENCE = 4
LR_FACTOR   = 0.5
SEEDS       = [42, 43, 44, 45, 46]
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Device  : {DEVICE}')
print(f'IMG_SIZE: {IMG_SIZE}')
print(f'BATCH   : {BATCH_SIZE}')

In [5]:
import os
if IN_COLAB:
    os.makedirs('/content/Imgs', exist_ok=True)

    n_local = len(os.listdir('/content/Imgs'))
    if n_local < 100:
        zip_path1 = '/content/drive/MyDrive/Trilha3-V2/Data/Imgs.zip'
        zip_path2 = '/content/drive/MyDrive/Trilha3-V2/Data/Imgs/Imgs.zip'
        zip_path3 = '/content/drive/MyDrive/Doutorado-V3/Data/Imgs.zip'

        source_zip = None
        for zp in [zip_path1, zip_path2, zip_path3]:
            if os.path.exists(zp):
                source_zip = zp
                break

        if source_zip:
            print(f'Descompactando imagens de {source_zip} direto no SSD...')
            os.system(f'unzip -q -o {source_zip} -d /content/Imgs/')

            if os.path.exists('/content/Imgs/Imgs'):
                os.system('mv /content/Imgs/Imgs/* /content/Imgs/')
                os.system('rm -rf /content/Imgs/Imgs')

            n_local = len(os.listdir('/content/Imgs'))
            print(f'Pronto em segundos! Total de {n_local} imagens transferidas.')
        else:
            print('Arquivo Imgs.zip não encontrado no Drive. Verifique se o upload terminou e se o nome está correto!')
    else:
        print(f'Tudo certo! SSD já possui {n_local} imagens prontas para treinar.')
else:
    BASE_DIR = Path(r'E:\Doutorado-V3\Trilha3-V2')
    IMGS_DIR = Path(r'E:\Doutorado-V3\Data\Imgs')

## 2. Dataset and transforms

In [6]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def get_transforms(split: str, augmentation: str = 'aug2') -> T.Compose:
    """
    split: 'train' | 'val' | 'test'
    augmentation: 'aug1' (flip só) | 'aug2' (moderada)
    Val e test nunca têm augmentation — o parâmetro só afeta treino.
    """
    normalize = T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)

    if split == 'train':
        if augmentation == 'aug1':
            return T.Compose([
                T.Resize((IMG_SIZE, IMG_SIZE)),
                T.RandomHorizontalFlip(p=0.5),
                T.ToTensor(),
                normalize,
            ])
        else:
            return T.Compose([
                T.Resize((int(IMG_SIZE * 1.15), int(IMG_SIZE * 1.15))),
                T.RandomHorizontalFlip(p=0.5),
                T.RandomRotation(degrees=10),
                T.RandomResizedCrop(size=IMG_SIZE, scale=(0.85, 1.0), ratio=(0.9, 1.1)),
                T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
                T.ToTensor(),
                normalize,
            ])
    else:
        return T.Compose([
            T.Resize((int(IMG_SIZE * 1.14), int(IMG_SIZE * 1.14))),
            T.CenterCrop(IMG_SIZE),
            T.ToTensor(),
            normalize,
        ])

class GastroscopyDataset(Dataset):
    def __init__(self, df: pd.DataFrame, imgs_dir: Path, transform=None):
        self.df        = df.reset_index(drop=True)
        self.imgs_dir  = imgs_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row    = self.df.iloc[idx]
        img    = Image.open(self.imgs_dir / row['image_name']).convert('RGB')
        labels = torch.tensor(row[CORE_LABELS].values.astype(float), dtype=torch.float32)
        if self.transform:
            img = self.transform(img)
        return img, labels

def build_weighted_sampler(df: pd.DataFrame) -> WeightedRandomSampler:
    """
    Peso de cada amostra = inverso da freq da sua classe positiva mais rara.
    Garante que exemplos de MICRONODULARIDADE e PÓLIPO apareçam proporcionalmente.
    """
    freq = df[CORE_LABELS].mean(axis=0).values
    inv_freq = 1.0 / (freq + 1e-6)

    weights = []
    for _, row in df.iterrows():
        active = np.where(row[CORE_LABELS].values == 1)[0]
        w = inv_freq[active].max() if len(active) > 0 else 1.0
        weights.append(w)

    return WeightedRandomSampler(
        weights=weights,
        num_samples=len(weights),
        replacement=True
    )

def compute_pos_weights(df: pd.DataFrame) -> torch.Tensor:
    """pos_weight = N_neg / N_pos por classe, para BCEWithLogitsLoss."""
    pos = df[CORE_LABELS].sum(axis=0).values
    neg = len(df) - pos
    w   = neg / (pos + 1e-6)
    return torch.tensor(w, dtype=torch.float32).to(DEVICE)

print('Dataset e transforms definidos.')

## 3. Models

In [7]:
def build_model(backbone_name: str, n_classes: int = N_CLASSES) -> nn.Module:
    """
    Retorna modelo com cabeça multilabel substituída.
    backbone_name: 'resnet50' | 'efficientnet_b3' | 'convnext_tiny' | 'swin_tiny'
    """
    if backbone_name == 'resnet50':
        model = tvm.resnet50(weights=tvm.ResNet50_Weights.IMAGENET1K_V2)
        model.fc = nn.Linear(model.fc.in_features, n_classes)

    elif backbone_name == 'efficientnet_b3':
        model = timm.create_model('efficientnet_b3', pretrained=True, num_classes=n_classes)

    elif backbone_name == 'convnext_tiny':
        model = timm.create_model('convnext_tiny.fb_in22k_ft_in1k',
                                  pretrained=True, num_classes=n_classes)

    elif backbone_name == 'swin_tiny':
        model = timm.create_model('swin_tiny_patch4_window7_224.ms_in22k_ft_in1k',
                                  pretrained=True, num_classes=n_classes)

    else:
        raise ValueError(f'backbone desconhecido: {backbone_name}')

    return model

def build_optimizer(model: nn.Module, backbone_name: str):
    """
    LR diferenciado: backbone = LR_BACKBONE, cabeça = LR_HEAD.
    Swin-Tiny usa o mesmo esquema — 'head' é o nome da camada final no timm.
    """
    if backbone_name == 'resnet50':
        head_params     = list(model.fc.parameters())
        backbone_params = [p for n, p in model.named_parameters() if 'fc' not in n]
    elif backbone_name == 'efficientnet_b3':
        head_params     = list(model.classifier.parameters())
        backbone_params = [p for n, p in model.named_parameters() if 'classifier' not in n]
    elif backbone_name in ('convnext_tiny', 'swin_tiny'):
        head_params     = list(model.head.parameters())
        backbone_params = [p for n, p in model.named_parameters() if 'head' not in n]

    return torch.optim.AdamW(
        [
            {'params': backbone_params, 'lr': LR_BACKBONE},
            {'params': head_params,     'lr': LR_HEAD},
        ],
        weight_decay=WEIGHT_DECAY,
    )

print('Funções de modelo definidas.')

## 4. Metrics

In [8]:
def optimize_thresholds(probs: np.ndarray, labels: np.ndarray) -> np.ndarray:
    """Encontra threshold por classe que maximiza F1 individual em validação."""
    thresholds = np.zeros(N_CLASSES)
    for i in range(N_CLASSES):
        best_t, best_f1 = 0.5, 0.0
        for t in np.arange(0.1, 0.9, 0.05):
            preds = (probs[:, i] >= t).astype(int)
            if preds.sum() == 0:
                continue
            f1 = f1_score(labels[:, i], preds, zero_division=0)
            if f1 > best_f1:
                best_f1, best_t = f1, t
        thresholds[i] = best_t
    return thresholds

def compute_metrics(probs: np.ndarray, labels: np.ndarray,
                    thresholds: np.ndarray = None) -> dict:
    """Calcula todas as métricas de avaliação."""
    if thresholds is None:
        thresholds = np.full(N_CLASSES, 0.5)

    preds = (probs >= thresholds).astype(int)

    f1_per_class  = f1_score(labels, preds, average=None, zero_division=0)
    pr_per_class  = precision_score(labels, preds, average=None, zero_division=0)
    rc_per_class  = recall_score(labels, preds, average=None, zero_division=0)

    auprc = []
    for i in range(N_CLASSES):
        if labels[:, i].sum() > 0:
            auprc.append(average_precision_score(labels[:, i], probs[:, i]))
        else:
            auprc.append(float('nan'))

    roc_auc = []
    for i in range(N_CLASSES):
        if labels[:, i].sum() > 0 and labels[:, i].sum() < len(labels):
            roc_auc.append(roc_auc_score(labels[:, i], probs[:, i]))
        else:
            roc_auc.append(float('nan'))

    f1_macro = float(np.nanmean(f1_per_class))
    f1_micro = float(f1_score(labels, preds, average='micro', zero_division=0))

    hamming = float(np.mean(preds != labels))

    multi_mask = labels.sum(axis=1) >= 2
    if multi_mask.sum() > 0:
        f1_multi = float(f1_score(labels[multi_mask], preds[multi_mask],
                                   average='macro', zero_division=0))
    else:
        f1_multi = float('nan')

    result = {
        'f1_macro':  f1_macro,
        'f1_micro':  f1_micro,
        'hamming':   hamming,
        'f1_multi':  f1_multi,
        'pr_auc_macro': float(np.nanmean(auprc)),
        'roc_auc_macro': float(np.nanmean(roc_auc)),
    }

    for i, label in enumerate(CORE_LABELS):
        result[f'f1_{label}']     = float(f1_per_class[i])
        result[f'prec_{label}']   = float(pr_per_class[i])
        result[f'rec_{label}']    = float(rc_per_class[i])
        result[f'auprc_{label}']  = float(auprc[i])
        result[f'rocauc_{label}'] = float(roc_auc[i])
        result[f'thr_{label}']    = float(thresholds[i])

    return result

print('Funções de métricas definidas.')

## 5. Training loop

In [9]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark     = True

def log_print(msg=""):
    print(msg)
    with open(RESULTS_DIR / 'training_log.txt', 'a', encoding='utf-8') as f:
        f.write(str(msg) + '\n')

@torch.no_grad()
def evaluate(model, loader):
    """Retorna probabilidades e rótulos verdadeiros."""
    model.eval()
    all_probs, all_labels = [], []
    for i, (imgs, labels) in enumerate(loader):
        imgs = imgs.to(DEVICE)
        logits = model(imgs)
        probs  = torch.sigmoid(logits).cpu().numpy()
        all_probs.append(probs)
        all_labels.append(labels.numpy())
        if RUN_FAST_CHECK and i >= 2:
            break
    return np.concatenate(all_probs), np.concatenate(all_labels)

def train_one_fold(fold: int, backbone_name: str, augmentation: str,
                   save_checkpoint: bool = True, verbose: bool = True) -> dict:
    """
    Treina um fold completo. Retorna métricas de teste.
    """
    seed = SEEDS[fold]
    set_seed(seed)

    df_tr  = pd.read_csv(SPLITS_DIR / f'fold_{fold}_train.csv')
    df_val = pd.read_csv(SPLITS_DIR / f'fold_{fold}_val.csv')
    df_te  = pd.read_csv(SPLITS_DIR / f'fold_{fold}_test.csv')

    tf_tr  = get_transforms('train', augmentation)
    tf_ev  = get_transforms('val')

    ds_tr  = GastroscopyDataset(df_tr,  IMGS_DIR, tf_tr)
    ds_val = GastroscopyDataset(df_val, IMGS_DIR, tf_ev)
    ds_te  = GastroscopyDataset(df_te,  IMGS_DIR, tf_ev)

    workers = 0
    dl_tr  = DataLoader(ds_tr,  batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=workers, pin_memory=True)
    dl_val = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=workers, pin_memory=True)
    dl_te  = DataLoader(ds_te,  batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=workers, pin_memory=True)

    model     = build_model(backbone_name).to(DEVICE)

    ckpt_name = f'{backbone_name}_{augmentation}_fold{fold}.pt'
    ckpt_path = CKPT_DIR / ckpt_name
    if ckpt_path.exists():
        if verbose: log_print(f'-> [RESUMO ATIVADO] Encontrado checkpoint {ckpt_name}! Carregando e pulando treinamento.')
        model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
        v_probs, v_labels = evaluate(model, dl_val)
        best_thr = optimize_thresholds(v_probs, v_labels)
        val_metrics = compute_metrics(v_probs, v_labels, best_thr)
        t_probs, t_labels = evaluate(model, dl_te)
        test_metrics = compute_metrics(t_probs, t_labels, best_thr)
        test_metrics['fold'] = fold
        test_metrics['best_val_f1'] = val_metrics['f1_macro']
        test_metrics['epochs_trained'] = 0
        test_metrics['val_thresholds'] = best_thr.tolist()
        test_metrics['history'] = {'train_loss': [], 'val_f1': []}
        return test_metrics

    pos_w     = compute_pos_weights(df_tr)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    optimizer = build_optimizer(model, backbone_name)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', patience=LR_PATIENCE, factor=LR_FACTOR
    )

    best_val_f1  = -1.0
    best_weights = None
    es_counter   = 0
    history      = {'train_loss': [], 'val_f1': []}

    for epoch in range(1, MAX_EPOCHS + 1):

        model.train()
        running_loss = 0.0
        pbar = tqdm(dl_tr, desc=f'Fold {fold} | Epoch {epoch}', leave=False) if verbose else dl_tr
        for i, (imgs, labels) in enumerate(pbar):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()

            loss = criterion(model(imgs), labels)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
            optimizer.step()

            running_loss += loss.item() * len(imgs)

            if RUN_FAST_CHECK and i >= 2:
                break

        train_loss = running_loss / len(ds_tr)

        val_probs, val_labels = evaluate(model, dl_val)
        val_thr  = optimize_thresholds(val_probs, val_labels)
        val_met  = compute_metrics(val_probs, val_labels, val_thr)
        val_f1   = val_met['f1_macro']

        scheduler.step(val_f1)
        history['train_loss'].append(train_loss)
        history['val_f1'].append(val_f1)

        if verbose and (epoch % 5 == 0 or epoch == 1):
            log_print(f'  Epoch {epoch:>3} | loss={train_loss:.4f} | val_F1={val_f1:.4f}')

        if val_f1 > best_val_f1 + 1e-4:
            best_val_f1  = val_f1
            best_weights = deepcopy(model.state_dict())
            es_counter   = 0
        else:
            es_counter += 1
            if es_counter >= ES_PATIENCE:
                if verbose:
                    log_print(f'  Early stopping na época {epoch}.')
                break

    model.load_state_dict(best_weights)
    if save_checkpoint:
        ckpt_name = f'{backbone_name}_{augmentation}_fold{fold}.pt'
        torch.save(best_weights, CKPT_DIR / ckpt_name)

    test_probs, test_labels = evaluate(model, dl_te)
    test_met = compute_metrics(test_probs, test_labels, val_thr)

    test_met['fold']            = fold
    test_met['best_val_f1']     = best_val_f1
    test_met['epochs_trained']  = len(history['train_loss'])
    test_met['val_thresholds']  = val_thr.tolist()
    test_met['history']         = history

    return test_met

log_print('Loop de treino definido com TQDM e AMP.')

## 6. Execution — all experiments

> **Estimated time (A100):** ~10–15 min per experiment · 5 experiments × 5 folds = ~4–6h total.  
> To test the pipeline, set `RUN_FAST_CHECK = True` — trains only 1 fold per experiment with 5 epochs.

In [10]:
RUN_FAST_CHECK = False

EXPERIMENTS = [
    {'id': 'resnet50_aug1',    'backbone': 'resnet50',        'aug': 'aug1'},
    {'id': 'resnet50_aug2',    'backbone': 'resnet50',        'aug': 'aug2'},
    {'id': 'efficientnet_b3',  'backbone': 'efficientnet_b3', 'aug': 'aug2'},
    {'id': 'convnext_tiny',    'backbone': 'convnext_tiny',   'aug': 'aug2'},
    {'id': 'swin_tiny',        'backbone': 'swin_tiny',       'aug': 'aug2'},
]

if RUN_FAST_CHECK:
    MAX_EPOCHS   = 5
    FOLDS_TO_RUN = [0]
    print('MODO FAST CHECK: 1 fold, 5 épocas.')
else:
    FOLDS_TO_RUN = list(range(N_FOLDS))
    print(f'MODO COMPLETO: {N_FOLDS} folds, até {MAX_EPOCHS} épocas.')
    print(f'Experimentos: {[e["id"] for e in EXPERIMENTS]}')

In [11]:
all_results = {}

for exp in EXPERIMENTS:
    exp_id = exp['id']
    backbone = exp['backbone']
    aug = exp['aug']

    log_print(f'\n{"=" * 60}')
    log_print(f'Experimento: {exp_id}  (backbone={backbone}, aug={aug})')
    log_print(f'{"=" * 60}')

    fold_results = []
    t0 = time.time()

    for fold in FOLDS_TO_RUN:
        log_print(f'\n  Fold {fold}/{N_FOLDS - 1}')

        met = train_one_fold(
            fold,
            backbone,
            aug,
            save_checkpoint=not RUN_FAST_CHECK
        )

        fold_results.append(met)

        log_print(
            f'  → Test Macro-F1={met["f1_macro"]:.4f}  '
            f'PÓLIPO={met["f1_PÓLIPO"]:.4f}  '
            f'MICRONOD={met["f1_MICRONODULARIDADE"]:.4f}'
        )

    elapsed = time.time() - t0

    metric_keys = (
        ['f1_macro', 'f1_micro', 'hamming', 'f1_multi', 'pr_auc_macro', 'roc_auc_macro']
        + [f'f1_{l}' for l in CORE_LABELS]
        + [f'auprc_{l}' for l in CORE_LABELS]
    )

    agg = {}

    for k in metric_keys:
        vals = [
            r[k]
            for r in fold_results
            if not np.isnan(r.get(k, float('nan')))
        ]

        if vals:
            agg[k] = {
                'mean': float(np.mean(vals)),
                'std': float(np.std(vals))
            }

    all_results[exp_id] = {
        'config': exp,
        'fold_results': fold_results,
        'aggregated': agg,
        'elapsed_sec': round(elapsed, 1),
    }

    log_print(
        f'\n  AGREGADO: Macro-F1 = '
        f'{agg["f1_macro"]["mean"]:.4f} ± {agg["f1_macro"]["std"]:.4f}'
    )

    log_print(f'  Tempo total experimento: {elapsed / 60:.1f} min')

    with open(RESULTS_DIR / 'baseline_results.json', 'w', encoding='utf-8') as f:
        json.dump(all_results, f, ensure_ascii=False, indent=2)

    log_print('  Salvo em results/baseline/baseline_results.json')

## 7. Comparative table

In [12]:
TRILHA3_REF = {
    'f1_macro':           {'mean': 0.5171, 'std': 0.0449},
    'pr_auc_macro':       {'mean': 0.5840, 'std': 0.0310},
    'f1_ENANTEMA':        {'mean': 0.5378, 'std': 0.0375},
    'f1_PÓLIPO':          {'mean': 0.5065, 'std': 0.1095},
    'f1_ÚLCERA':          {'mean': 0.5200, 'std': 0.1009},
    'f1_EROSÃO':          {'mean': 0.6702, 'std': 0.0201},
    'f1_MICRONODULARIDADE':{'mean': 0.3513, 'std': 0.1845},
}

rows = []

row = {'Modelo': 'Trilha-3 (ResNet50 AUG1)'}
for k, label in [('f1_macro','F1-macro'), ('pr_auc_macro','PR-AUC'),
                  ('f1_ENANTEMA','ENANTEMA'), ('f1_PÓLIPO','PÓLIPO'),
                  ('f1_ÚLCERA','ÚLCERA'), ('f1_EROSÃO','EROSÃO'),
                  ('f1_MICRONODULARIDADE','MICRONOD')]:
    v = TRILHA3_REF.get(k)
    row[label] = f"{v['mean']:.3f}±{v['std']:.3f}" if v else '-'
rows.append(row)

for exp_id, res in all_results.items():
    agg = res['aggregated']
    row = {'Modelo': exp_id}
    for k, label in [('f1_macro','F1-macro'), ('pr_auc_macro','PR-AUC'),
                      ('f1_ENANTEMA','ENANTEMA'), ('f1_PÓLIPO','PÓLIPO'),
                      ('f1_ÚLCERA','ÚLCERA'), ('f1_EROSÃO','EROSÃO'),
                      ('f1_MICRONODULARIDADE','MICRONOD')]:
        v = agg.get(k)
        row[label] = f"{v['mean']:.3f}±{v['std']:.3f}" if v else '-'
    rows.append(row)

df_table = pd.DataFrame(rows).set_index('Modelo')
print('=== TABELA COMPARATIVA ===')
print(df_table.to_string())

df_table.to_csv(RESULTS_DIR / 'tabela_comparativa.csv')
print('\nTabela salva em results/baseline/tabela_comparativa.csv')

## 8. Figure — F1-macro per experiment

In [13]:
def plot_f1_macro(lang='pt'):
    fig, ax = plt.subplots(figsize=(9, 4))

    exp_ids  = ['Trilha-3\n(ResNet50 AUG1)'] + list(all_results.keys())
    means    = [TRILHA3_REF['f1_macro']['mean']] + [all_results[e]['aggregated']['f1_macro']['mean'] for e in all_results]
    stds     = [TRILHA3_REF['f1_macro']['std']] + [all_results[e]['aggregated']['f1_macro']['std'] for e in all_results]

    colors   = ['#aaaaaa'] + ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0'][:len(all_results)]
    bars = ax.bar(exp_ids, means, yerr=stds, capsize=4, color=colors, edgecolor='white', linewidth=0.8)

    for bar, m, s in zip(bars, means, stds):
        ax.text(bar.get_x() + bar.get_width()/2, m + s + 0.005,
                f'{m:.3f}', ha='center', va='bottom', fontsize=8.5, fontweight='bold')

    if lang == 'pt':
        ax.axhline(TRILHA3_REF['f1_macro']['mean'], color='gray', linestyle='--', linewidth=1, alpha=0.7, label='Referência Trilha-3')
        ax.set_ylabel('Macro-F1 (média ± std, 5 folds)')
        ax.set_title('Comparação de Backbones — Trilha3-V2', fontweight='bold')
    else:
        ax.axhline(TRILHA3_REF['f1_macro']['mean'], color='gray', linestyle='--', linewidth=1, alpha=0.7, label='Track-3 Baseline')
        ax.set_ylabel('Macro-F1 (mean ± std, 5 folds)')
        ax.set_title('Backbone Comparison — Track3-V2', fontweight='bold')

    ax.set_ylim(0, min(1.0, max(means) + max(stds) + 0.08))
    ax.legend(fontsize=8)
    plt.tight_layout()
    fig.savefig(FIGS_DIR / f'backbone_comparison_f1macro_{lang}.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_f1_macro('pt')
plot_f1_macro('en')
log_print('Figuras (F1-Macro) salvas em PT e EN.')

## 9. Figure — Heatmap F1 per class and experiment

In [14]:
def plot_heatmap(lang='pt'):
    heat_data = {'Trilha-3 (ref)' if lang == 'pt' else 'Track-3 (ref)': [TRILHA3_REF[f'f1_{l}']['mean'] for l in CORE_LABELS]}
    for exp_id, res in all_results.items():
        heat_data[exp_id] = [res['aggregated'].get(f'f1_{l}', {}).get('mean', float('nan')) for l in CORE_LABELS]

    labels_en = {'ENANTEMA':'ERYTHEMA', 'PÓLIPO':'POLYP', 'ÚLCERA':'ULCER', 'EROSÃO':'EROSION', 'MICRONODULARIDADE':'MICRONODULARITY'}
    y_labels = CORE_LABELS if lang == 'pt' else [labels_en.get(l, l) for l in CORE_LABELS]

    df_heat = pd.DataFrame(heat_data, index=y_labels).T
    fig, ax = plt.subplots(figsize=(9, 0.7 * len(df_heat) + 1.5))
    sns.heatmap(df_heat, annot=True, fmt='.3f', cmap='RdYlGn',
                vmin=0.2, vmax=0.85, linewidths=0.4,
                ax=ax, cbar_kws={'label': 'F1'})

    ax.set_title('F1 por classe e modelo' if lang == 'pt' else 'F1 per class and model', fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('')
    plt.tight_layout()
    fig.savefig(FIGS_DIR / f'f1_heatmap_backbones_{lang}.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_heatmap('pt')
plot_heatmap('en')
log_print('Figuras (Heatmap) salvas em PT e EN.')

## 10. Final summary

In [15]:
log_print('=== RESUMO FINAL ===')
log_print(f'Referência Trilha-3 : Macro-F1 = 0.517 ± 0.045')
log_print()

best_exp, best_f1 = None, -1.0
for exp_id, res in all_results.items():
    m = res['aggregated']['f1_macro']['mean']
    s = res['aggregated']['f1_macro']['std']
    delta = m - 0.5171
    flag  = ' ← MELHOR' if m > best_f1 else ''
    if m > best_f1:
        best_f1, best_exp = m, exp_id
    log_print(f'{exp_id:<22} Macro-F1 = {m:.4f} ± {s:.4f}  (Δ={delta:+.4f}){flag}')

log_print(f'\nMelhor backbone identificado: {best_exp} (F1={best_f1:.4f})')
log_print('Este backbone será usado no NB03 (M2 otimizado).')

In [16]:
import time
if IN_COLAB:
    log_print('Pipeline concluído! A sessão do Colab será finalizada em 5 minutos para poupar créditos.')
    time.sleep(300)
    from google.colab import runtime
    runtime.unassign()
else:
    log_print('Pipeline concluído! (Ambiente local, não será encerrado).')